# Implement PPO for RLHF — Solution

**Difficulty**: 🔴 Hard

**Companies**: Anthropic, OpenAI, DeepMind, Meta

---

### Problem Statement

Proximal Policy Optimization (PPO) is the workhorse algorithm behind RLHF (Reinforcement Learning from Human Feedback), used to train models like ChatGPT and Claude. In the RLHF pipeline, a reward model (trained on human preferences) provides scalar rewards, and PPO optimizes the policy to maximize those rewards while staying close to a reference policy.

PPO uses a clipped surrogate objective to prevent the policy from changing too drastically in a single update, along with a value function baseline to reduce variance, and a KL penalty to keep the policy close to the reference.

**Where does the KL penalty go?** The RLHF objective is `E[r_φ(x, y)] − β·KL(π_θ ‖ π_ref)`, and there are two standard places to put the `β·KL` term:

- **In the reward** (InstructGPT, TRL's `PPOTrainer`): every token gets `r_t = −β·(log π_old(a_t|s_t) − log π_ref(a_t|s_t))`, the reward-model score is added at the last token, and GAE runs on these shaped rewards.
- **In the loss** (GRPO, OpenRLHF's `use_kl_loss`): add `β·KL̂` to the PPO loss. Here the estimator matters. The naive per-token estimate `log π_θ − log π_ref` has the right *value* on average, but as a loss its expected gradient is `E_{a∼π_θ}[∇ log π_θ(a)] = 0`, so it regularizes nothing. Use `k3 = π_ref/π_θ − log(π_ref/π_θ) − 1` instead: it is non-negative, unbiased for the KL, and its gradient pulls `π_θ` toward `π_ref`. (Precisely: as a loss, k3's expected gradient is that of the *reverse* KL(π_ref ‖ π_θ), which agrees with ∇KL(π_θ ‖ π_ref) to first order near the reference. The per-token estimator whose expected gradient is exactly ∇KL(π_θ ‖ π_ref) is k2 = ½(log π_θ − log π_ref)².)

This problem uses the loss-term placement (`ppo_step` takes `kl_beta`).

Your task is to implement the rollout, GAE, and the PPO update (clipped surrogate + value loss + KL penalty), then train on a toy task.

---

### Requirements

The policy, value and reward models are provided. You implement:

1. **generate_sequences** — Sample rollouts token by token from the policy, starting from a BOS token (id = `vocab_size`), and record each sampled token's log-prob under the sampling policy. Also return the BOS-prefixed token ids `(batch, seq_len + 1)` so the value model and `ppo_step` can score the same states.
2. **compute_gae** — Generalized Advantage Estimation: a backward recursion over per-token rewards and value estimates, with V = 0 after the last token.
3. **ppo_step** — The PPO clipped surrogate loss, the value-function MSE, and a KL penalty against the frozen reference policy, combined as `total_loss = policy_loss + 0.5·value_loss + kl_beta·kl_div`.

---

### Constraints

- ✅ Use a simple task: generate sequences that sum to a target value.
- ✅ Include KL penalty against a reference policy.
- ✅ Average reward should increase over training.
- ✅ KL divergence from reference should stay bounded.
- ❌ Do **not** use any existing RL/RLHF libraries (trl, stable-baselines, etc.).

---

<details>
  <summary>💡 Hint</summary>

  - GAE: A_t = delta_t + (gamma * lam) * delta_{t+1} + ... where delta_t = r_t + gamma * V(s_{t+1}) - V(s_t), with V = 0 after the final token. Compute it backwards: A_t = delta_t + gamma * lam * A_{t+1}. The value targets are returns = A_t + V(s_t).
  - PPO clipped objective (to maximize): min(ratio * A, clip(ratio, 1-eps, 1+eps) * A) where ratio = pi(a|s) / pi_old(a|s) = exp(log_pi - log_pi_old). The old log-probs are constants recorded at rollout time. The `min` is pessimistic: with A < 0 a large ratio is *not* clipped.
  - KL penalty as a loss term: average k3 = exp(log_ref - log_pi) - (log_ref - log_pi) - 1 over tokens. (`log_pi - log_ref` is the right per-token penalty to subtract from the *reward*, and fine for logging, but as a loss term its expected gradient is zero.)
  - Process: (1) generate sequences from policy, (2) score with reward model, (3) compute GAE advantages and whiten them over the batch, (4) do multiple PPO epochs on the collected batch.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy

In [ ]:
# Task: generate sequences of discrete tokens.
# Reward = -|sum(tokens) - target|  (closer to target = higher reward)
VOCAB_SIZE = 10
SEQ_LEN = 8
TARGET_SUM = 36  # target sum of token values


class PolicyModel(nn.Module):
    """Policy network: takes a state (partial sequence) and outputs logits over vocab."""
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=32, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size + 1, embed_dim)  # +1 for BOS token
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_ids):
        """Returns logits for each position: (batch, seq_len, vocab_size)."""
        x = self.embedding(input_ids)
        h, _ = self.rnn(x)
        return self.head(h)


class ValueModel(nn.Module):
    """Value network: estimates expected future reward from current state."""
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=32, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size + 1, embed_dim)
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, input_ids):
        """Returns value estimates: (batch, seq_len)."""
        x = self.embedding(input_ids)
        h, _ = self.rnn(x)
        return self.head(h).squeeze(-1)  # (batch, seq_len)


class RewardModel(nn.Module):
    """Reward model: scores a complete sequence. Reward = -|sum(tokens) - target|."""
    def __init__(self, target_sum=TARGET_SUM):
        super().__init__()
        self.target_sum = target_sum

    def forward(self, sequences):
        """sequences: (batch, seq_len) of token IDs. Returns (batch,) rewards."""
        sums = sequences.float().sum(dim=-1)
        return -torch.abs(sums - self.target_sum)


torch.manual_seed(42)
policy = PolicyModel()
value_model = ValueModel()
ref_policy = copy.deepcopy(policy)
for p in ref_policy.parameters():
    p.requires_grad = False
reward_model = RewardModel()

print("Policy params:", sum(p.numel() for p in policy.parameters()))
print("Value params:", sum(p.numel() for p in value_model.parameters()))

In [ ]:
def generate_sequences(policy: nn.Module, batch_size: int, seq_len: int,
                        vocab_size: int) -> tuple:
    """
    Generate sequences autoregressively from the policy.

    Returns:
        sequences: (batch, seq_len) generated token IDs (the actions)
        log_probs: (batch, seq_len) log probability of each sampled token under policy
        all_input_ids: (batch, seq_len + 1) BOS followed by the generated tokens.
            Position t (BOS + tokens[:t]) is the state s_t in which action t was taken,
            so policy/value logits at positions 0..seq_len-1 line up with the actions.
    """
    # Start with BOS token (index = vocab_size)
    current = torch.full((batch_size, 1), vocab_size, dtype=torch.long)
    sequences, log_probs = [], []

    for _ in range(seq_len):
        # Re-encodes the whole prefix each step for clarity; a real implementation
        # would carry the RNN state / KV cache instead.
        last_logits = policy(current)[:, -1, :]  # (batch, vocab_size)
        dist = torch.distributions.Categorical(logits=last_logits)
        action = dist.sample()  # (batch,)
        sequences.append(action)
        log_probs.append(dist.log_prob(action))
        current = torch.cat([current, action.unsqueeze(1)], dim=1)

    sequences = torch.stack(sequences, dim=1)  # (batch, seq_len)
    log_probs = torch.stack(log_probs, dim=1)  # (batch, seq_len)
    return sequences, log_probs, current       # current: (batch, seq_len + 1)


def compute_gae(rewards: torch.Tensor, values: torch.Tensor,
                gamma: float = 0.99, lam: float = 0.95) -> torch.Tensor:
    """
    Compute Generalized Advantage Estimation.

    Args:
        rewards: (batch, seq_len) per-step rewards.
        values: (batch, seq_len) value estimates V(s_t) from value model.
        gamma: Discount factor.
        lam: GAE lambda.

    Returns:
        advantages: (batch, seq_len) advantage estimates.
    """
    batch_size, seq_len = rewards.shape
    advantages = torch.zeros_like(rewards)
    last_gae = torch.zeros_like(rewards[:, 0])

    for t in reversed(range(seq_len)):
        # The episode ends after the last token: V(s_T) = 0
        next_value = values[:, t + 1] if t + 1 < seq_len else torch.zeros_like(values[:, t])
        delta = rewards[:, t] + gamma * next_value - values[:, t]
        last_gae = delta + gamma * lam * last_gae
        advantages[:, t] = last_gae

    return advantages


def ppo_step(policy: nn.Module, value_model: nn.Module, ref_policy: nn.Module,
             old_log_probs: torch.Tensor, input_ids: torch.Tensor,
             actions: torch.Tensor, advantages: torch.Tensor,
             returns: torch.Tensor, clip_epsilon: float = 0.2,
             kl_beta: float = 0.01) -> dict:
    """
    Compute the PPO clipped loss with a value loss and a KL penalty.

    input_ids is the (batch, seq_len + 1) BOS-prefixed rollout; the logits at
    position t score action t, so positions 0..seq_len-1 are used.
    old_log_probs are constants from the rollout (no gradient flows into them).
    """
    # Current policy log-probs of the actions that were actually taken
    action_logits = policy(input_ids)[:, :-1, :]  # (batch, seq_len, vocab_size)
    new_log_probs = torch.distributions.Categorical(logits=action_logits).log_prob(actions)

    # Reference policy log-probs (frozen)
    with torch.no_grad():
        ref_logits = ref_policy(input_ids)[:, :-1, :]
        ref_log_probs = torch.distributions.Categorical(logits=ref_logits).log_prob(actions)

    # Clipped surrogate objective (maximized, so its negation is the loss)
    ratio = torch.exp(new_log_probs - old_log_probs.detach())  # (batch, seq_len)
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1 - clip_epsilon, 1 + clip_epsilon) * advantages
    policy_loss = -torch.min(surr1, surr2).mean()

    # Value loss: regress V(s_t) onto the GAE returns
    values = value_model(input_ids)[:, :-1]  # (batch, seq_len)
    value_loss = F.mse_loss(values, returns)

    # KL(pi || pi_ref) with the k3 estimator: ratio - log(ratio) - 1, ratio = pi_ref / pi.
    # Non-negative, unbiased, and its gradient pulls pi toward pi_ref. (The naive
    # mean(log pi - log pi_ref) has zero expected gradient when used as a loss.)
    log_ratio_ref = ref_log_probs - new_log_probs
    kl_div = (torch.exp(log_ratio_ref) - log_ratio_ref - 1).mean()

    total_loss = policy_loss + 0.5 * value_loss + kl_beta * kl_div

    return {
        'policy_loss': policy_loss,
        'value_loss': value_loss,
        'kl_div': kl_div,
        'total_loss': total_loss
    }

In [ ]:
# === Validation ===
torch.manual_seed(42)

policy = PolicyModel()
value_model = ValueModel()
ref_policy = copy.deepcopy(policy)
for p in ref_policy.parameters():
    p.requires_grad = False
reward_model = RewardModel()

policy_optimizer = torch.optim.Adam(policy.parameters(), lr=3e-4)
value_optimizer = torch.optim.Adam(value_model.parameters(), lr=1e-3)

batch_size = 64
num_iterations = 30
ppo_epochs = 4

reward_history = []
kl_history = []

print("=== PPO Training ===")
for iteration in range(num_iterations):
    # 1. Generate sequences from current policy
    policy.eval()
    with torch.no_grad():
        sequences, old_log_probs, input_ids = generate_sequences(
            policy, batch_size, SEQ_LEN, VOCAB_SIZE)
        rewards_final = reward_model(sequences)  # (batch,)
        values = value_model(input_ids)[:, :-1]  # (batch, seq_len): V(s_t) before each action

    # Spread reward to last step only
    step_rewards = torch.zeros(batch_size, SEQ_LEN)
    step_rewards[:, -1] = rewards_final

    # 2. Compute GAE advantages
    advantages = compute_gae(step_rewards, values.detach())
    returns = advantages + values.detach()
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    # 3. PPO update epochs
    policy.train()
    value_model.train()
    for _ in range(ppo_epochs):
        loss_dict = ppo_step(policy, value_model, ref_policy,
                             old_log_probs, input_ids, sequences,
                             advantages, returns)

        policy_optimizer.zero_grad()
        value_optimizer.zero_grad()
        loss_dict['total_loss'].backward()
        policy_optimizer.step()
        value_optimizer.step()

    avg_reward = rewards_final.mean().item()
    avg_kl = loss_dict['kl_div'].item()
    reward_history.append(avg_reward)
    kl_history.append(avg_kl)

    if iteration % 5 == 0:
        print(f"Iter {iteration:3d} | Reward: {avg_reward:.2f} | KL: {avg_kl:.4f}")

# Test 1: Reward should increase
print("\n=== Reward Increase Test ===")
early_reward = sum(reward_history[:5]) / 5
late_reward = sum(reward_history[-5:]) / 5
print(f"Early avg reward: {early_reward:.2f}")
print(f"Late avg reward:  {late_reward:.2f}")
assert late_reward > early_reward, "Average reward should increase over training!"
print("PASSED\n")

# Test 2: KL should stay bounded
print("=== KL Bounded Test ===")
max_kl = max(kl_history)
print(f"Max KL divergence: {max_kl:.4f}")
assert max_kl < 50.0, "KL divergence should stay bounded!"
print("PASSED\n")

print("All tests passed!")